<a href="https://colab.research.google.com/github/djdubeyji/DoraRegistryApp/blob/main/DoraRegistryApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import csv
import os
import math

CSV_FILE = 'dora_registry.csv'
LAMBDA_LOSS = 2.25  # Loss Aversion (Prospect Theory)
ALPHA = 0.88        # Diminishing Sensitivity
VENDOR_MULTIPLIER = 0.15 # 15% risk increase per shared dependency

def calculate_prospect_score(recovery_cost, data_value, rto, vendor_count):
    """
    Formula:
    1. Base Loss = Cost + (DataValue * 0.1 * RTO)
    2. PT Score = -Lambda * (Base Loss ^ Alpha)
    3. Final Score = PT Score * (1 + (VendorMultiplier * (VendorCount - 1)))
       -> If multiple assets share a vendor, the risk amplifies.
    """
    # 1. Estimate Potential Economic Loss
    base_loss = float(recovery_cost) + (float(data_value) * 0.1 * float(rto))

    # 2. Apply Prospect Theory (Psychological Impact)
    pt_score = -LAMBDA_LOSS * (abs(base_loss) ** ALPHA)

    # 3. Apply Concentration Risk Multiplier (DORA Art. 28)
    concentration_factor = 1 + (VENDOR_MULTIPLIER * max(0, vendor_count - 1))

    return pt_score * concentration_factor

def generate_id(existing_ids):
    """Auto-generates DORA_0001, DORA_0002, etc."""
    if not existing_ids:
        return "DORA_0001"

    max_num = 0
    for asset_id in existing_ids:
        try:
            num = int(asset_id.split('_')[1])
            if num > max_num:
                max_num = num
        except:
            continue

    return f"DORA_{max_num + 1:04d}"

def load_data():
    """Reads CSV into a list of dictionaries."""
    if not os.path.exists(CSV_FILE):
        return []

    with open(CSV_FILE, mode='r', newline='') as f:
        reader = csv.DictReader(f)
        return list(reader)

def save_data(data):
    """Writes list of dictionaries back to CSV."""
    if not data:
        return

    fieldnames = ['ID', 'Name', 'Vendor', 'CIF', 'DataValue', 'RecCost', 'RTO', 'RiskScore']

    with open(CSV_FILE, mode='w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)

def update_risk_scores(data):
    """Recalculates scores including Vendor Concentration Logic."""
    # Count Vendor Occurrences
    vendor_counts = {}
    for row in data:
        v = row['Vendor']
        vendor_counts[v] = vendor_counts.get(v, 0) + 1

    # Update Scores
    for row in data:
        count = vendor_counts[row['Vendor']]
        score = calculate_prospect_score(
            row['RecCost'], row['DataValue'], row['RTO'], count
        )
        row['RiskScore'] = f"{score:.2f}"

    return data

# --- MENU FUNCTIONS ---

def view_assets(data):
    if not data:
        print("\n[!] Registry is empty.")
        return

    print("\n--- VIEW OPTIONS ---")
    print("1. Sort by Risk (Descending - Most Risky First)")
    print("2. Sort by Risk (Ascending - Least Risky First)")
    choice = input("Select: ")

    # Re-calculate scores before showing to ensure accuracy
    data = update_risk_scores(data)

    reverse = True if choice == '1' else False
    sorted_data = sorted(data, key=lambda x: float(x['RiskScore']), reverse=reverse)
    # High Risk = High Negative Number.


    # Correction for display logic:
    if choice == '1':
        sorted_data = sorted(data, key=lambda x: float(x['RiskScore'])) # Default sort puts -10000 before -10
    else:
        sorted_data = sorted(data, key=lambda x: float(x['RiskScore']), reverse=True)

    print(f"\n{'ID':<10} {'Name':<20} {'Vendor':<15} {'Risk Score (Utils)':<20} {'Concentration'}")
    print("-" * 80)

    # Calculate counts again for display
    v_counts = {}
    for r in data: v_counts[r['Vendor']] = v_counts.get(r['Vendor'], 0) + 1

    for row in sorted_data:
        c_risk = "HIGH" if v_counts[row['Vendor']] > 1 else "Normal"
        print(f"{row['ID']:<10} {row['Name']:<20} {row['Vendor']:<15} {row['RiskScore']:<20} {c_risk}")

def add_asset(data):
    print("\n--- ADD NEW ASSET ---")
    name = input("Asset Name: ")
    vendor = input("Vendor (e.g. AWS, Microsoft): ")
    cif = input("Is this a Critical Function? (y/n): ").lower() == 'y'

    try:
        val = float(input("Data Value ($ estimate): "))
        cost = float(input("Recovery Cost ($ estimate): "))
        rto = float(input("RTO (Hours): "))
    except ValueError:
        print("[!] Invalid number entered.")
        return data

    new_id = generate_id([d['ID'] for d in data])

    new_record = {
        'ID': new_id,
        'Name': name,
        'Vendor': vendor,
        'CIF': str(cif),
        'DataValue': val,
        'RecCost': cost,
        'RTO': rto,
        'RiskScore': 0
    }

    data.append(new_record)
    print(f"[+] Asset {new_id} added successfully.")
    return data

def edit_asset(data):
    target_id = input("Enter Asset ID to edit (e.g. DORA_0001): ")
    for row in data:
        if row['ID'] == target_id:
            print(f"Editing {row['Name']} (Current Vendor: {row['Vendor']})")
            row['Name'] = input(f"New Name [{row['Name']}]: ") or row['Name']
            row['Vendor'] = input(f"New Vendor [{row['Vendor']}]: ") or row['Vendor']
            print("[+] Updated.")
            return data
    print("[!] Asset not found.")
    return data

#main

def main():
    data = load_data()

    while True:
        print("\n=== DORA COMPLIANCE ASSET REGISTRY ===")
        print("1. View Risky Assets")
        print("2. Add New Asset")
        print("3. Edit Asset")
        print("4. Save & Exit")

        cmd = input(">>> ")

        if cmd == '1':
            view_assets(data)
        elif cmd == '2':
            data = add_asset(data)
            data = update_risk_scores(data) # Update scores immediately
            save_data(data) # Auto-save
        elif cmd == '3':
            data = edit_asset(data)
            data = update_risk_scores(data)
            save_data(data)
        elif cmd == '4':
            data = update_risk_scores(data)
            save_data(data)
            print("[*] Data saved to CSV. Exiting.")
            break
        else:
            print("[!] Invalid option.")

if __name__ == "__main__":
    main()


=== DORA COMPLIANCE ASSET REGISTRY ===
1. View Risky Assets
2. Add New Asset
3. Edit Asset
4. Save & Exit
>>> 1

--- VIEW OPTIONS ---
1. Sort by Risk (Descending - Most Risky First)
2. Sort by Risk (Ascending - Least Risky First)
Select: 1

ID         Name                 Vendor          Risk Score (Utils)   Concentration
--------------------------------------------------------------------------------
DORA_0001  360t sql server      aws             -36054.18            Normal

=== DORA COMPLIANCE ASSET REGISTRY ===
1. View Risky Assets
2. Add New Asset
3. Edit Asset
4. Save & Exit


KeyboardInterrupt: Interrupted by user